# 03 — Anomaly Detection
Train an Isolation Forest and save the model artifacts.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
ROOT = Path.cwd().parent
RAW = ROOT/"data"/"raw"/"water_quality_raw.csv"
PROC = ROOT/"data"/"processed"
MODELS = ROOT/"models"

import joblib
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
clean=pd.read_csv(PROC/"water_quality_clean.csv")
Xdf=pd.read_csv(PROC/"water_quality_features.csv"); features=Xdf.columns.tolist()
scaler=StandardScaler().fit(clean[features]); X=scaler.transform(clean[features])
model=IsolationForest(n_estimators=200,contamination=.05,random_state=RANDOM_STATE,n_jobs=-1).fit(X)
MODELS.mkdir(parents=True,exist_ok=True); joblib.dump(model,MODELS/"anomaly_detector.pkl"); joblib.dump(scaler,MODELS/"scaler.pkl")
meta={"project":"AquaGuard AI","algorithm":"Isolation Forest","features":features,"contamination":.05,"random_state":RANDOM_STATE,
      "output_mapping":{"1":"normal","-1":"anomaly"},"note":"Decision support only; verify real-world water quality appropriately."}
(MODELS/"model_metadata.json").write_text(json.dumps(meta,indent=2),encoding="utf-8")


In [ ]:
pred=model.predict(X); scores=model.decision_function(X)
results=clean.copy(); results["anomaly_label"]=np.where(pred==-1,"anomaly","normal"); results["anomaly_score"]=scores
print(results["anomaly_label"].value_counts()); display(results.sort_values("anomaly_score").head(10))


In [ ]:
results["anomaly_label"].value_counts().plot(kind="bar"); plt.title("Normal vs Anomaly"); plt.tight_layout(); plt.show()
